In [52]:
## IMPORT

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf           # for OLS (smf.ols)
from sklearn.metrics import mean_squared_error  # perform mean-squared for RMSE
from statsmodels.stats.anova import anova_lm    # perform Chi-Squared test

## LOAD DATA

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_PCA_for_test.csv")
# test = pd.read_csv("data/GDP_Forecast_test.csv")

## DEFINE FUNCTIONS FOR REUSE

def calc_performance(formula1, model1, formula2, model2):
    print(pd.DataFrame({'model'   : [formula1, formula2],
                    'Adj.R^2' : [model1.rsquared_adj, model2.rsquared_adj]}))
    print()
    print(anova_lm(model1, model2, test = "Chisq"))

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

## CONVERT YEAR-QUARTER: SPLIT COLUMNS
# we are using a different dataset for test because we want to calculate RMSE before submitting

# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]

## USING SEPARATE TEST DATASET FOR TESTING RMSE

# convert the data to YQ, year and quarter
test["observation_date"] = pd.to_datetime(test["observation_date"], dayfirst = True)
test["year"] = test["observation_date"].dt.year
test["qtr"] = "Q" + test["observation_date"].dt.quarter.astype(str)
test["YQ"] = test["year"].astype(str) + test["qtr"]
# remove observation_date column and move GDP to the back while changing col name to NGDP
test.drop(columns = "observation_date", inplace = True)
col = test.pop("GDP_PCA")
test["NGDP"] = col
# get the quarterly
test.sort_values(by = "YQ")
# # check data for 1990 - 2015 is similar to train to continue with this dataset for test
# test[test["year"] <= 2015].transpose()
## NOTE: data from later years is not as accurate, but we will use this to tentatively test our RMSE
test = test[test["year"]>= 2016].reset_index(drop = True)
test.head()

,year,qtr,YQ,NGDP
0,2016,Q1,2016Q1,2.0
1,2016,Q2,2016Q2,4.1
2,2016,Q3,2016Q3,3.9
3,2016,Q4,2016Q4,4.2
4,2017,Q1,2017Q1,4.1


Types of models:
- OLS (smf.ols)

Things to consider:
- p-value: <0.05 (and coefficient fits mental model)
- r-squared: explained variation / total variation (high R^2 = model fits well)
- adjusted r-squared: downweights R^2 to make it unbiased (variation)
- test statistics
    - test coeff: t-test (more common, unknown pop_n s.d.) or z-test (known pop_n s.d)
    - test model as a whole: f-test (check adj. R^2 as well)
    - testing across models: chi-squared (chisq)
    - testing across models under ML concept: prediction errors - RMSE and MAE

In [53]:
mod1 = smf.ols("NGDP ~ year + qtr", train).fit()
print(mod1.summary())
# getting prediction values based off model
y_pred, y_true = mod1.predict(test), test["NGDP"]
print(f"\nRMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     3.324
Date:                Thu, 05 Mar 2026   Prob (F-statistic):             0.0134
Time:                        01:08:30   Log-Likelihood:                -244.91
No. Observations:                 104   AIC:                             499.8
Df Residuals:                      99   BIC:                             513.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    219.8404     68.424      3.213      0.0